In [1]:
import os
import json
import glob
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from src.augmentation import AugmentedDenoiseDataset

In [2]:
clean_files_aug = glob.glob(r"C:\Users\Ravin\PycharmProjects\vokie\data\clean_raw_for_augment\*.wav")
noise_files_aug = glob.glob(r"C:\Users\Ravin\Downloads\audio data\ESC-50-master\audio\*.wav")
rir_files_aug = glob.glob(r"C:\Users\Ravin\Downloads\audio data (2)\RIRS_NOISES\simulated_rirs\**\*.wav", recursive=True)

print("Clean files:", len(clean_files_aug))
print("Noise files:", len(noise_files_aug))
print("RIR files:", len(rir_files_aug))

train_size_aug = int(0.85 * len(clean_files_aug))
train_clean_aug = clean_files_aug[:train_size_aug]
val_clean_aug = clean_files_aug[train_size_aug:]

train_data_aug = AugmentedDenoiseDataset(train_clean_aug, noise_files_aug, rir_files_aug)
val_data_aug = AugmentedDenoiseDataset(val_clean_aug, noise_files_aug, rir_files_aug)

print("Train size:", len(train_data_aug))
print("Val size:", len(val_data_aug))

Clean files: 11572
Noise files: 2000
RIR files: 60000
Train size: 9836
Val size: 1736


In [3]:
class DenoiseGRU(nn.Module):
    def __init__(self, freq_bins=257, hidden_size=224, num_layers=2, dropout=0.2):
        super().__init__()
        self.gru = nn.GRU(
            input_size=freq_bins,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=False,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.fc = nn.Linear(hidden_size, freq_bins)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        out, _ = self.gru(x)
        mask = self.sigmoid(self.fc(out))
        return mask

device_aug = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device_aug)

cuda


In [4]:
os.makedirs("src/models", exist_ok=True)
os.makedirs("src/outputs", exist_ok=True)

HYPERPARAMS_AUG = {
    "freq_bins": 257,
    "hidden_size": 224,
    "num_layers": 2,
    "dropout": 0.2,
    "bidirectional": False,
    "batch_size": 32,
    "lr": 0.0005,
    "epochs": 40,
    "patience": 8
}

with open("outputs/best_params_augmented.json", "w") as f:
    json.dump(HYPERPARAMS_AUG, f, indent=4)
print("Saved parameters to outputs/best_params_augmented.json")

train_loader_aug = DataLoader(train_data_aug, batch_size=HYPERPARAMS_AUG["batch_size"], shuffle=True,num_workers=1)
val_loader_aug = DataLoader(val_data_aug, batch_size=HYPERPARAMS_AUG["batch_size"], shuffle=False, num_workers=4)

Saved parameters to outputs/best_params_augmented.json


In [5]:
final_model_aug = DenoiseGRU(
    freq_bins=HYPERPARAMS_AUG["freq_bins"],
    hidden_size=HYPERPARAMS_AUG["hidden_size"],
    num_layers=HYPERPARAMS_AUG["num_layers"],
    dropout=HYPERPARAMS_AUG["dropout"]
).to(device_aug)

criterion_aug = nn.MSELoss()
optimizer_aug = torch.optim.Adam(final_model_aug.parameters(), lr=HYPERPARAMS_AUG["lr"])
scheduler_aug = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer_aug, mode='min', factor=0.5, patience=3)

EPOCHS_AUG = HYPERPARAMS_AUG["epochs"]

In [6]:
train_losses_aug = []
val_losses_aug = []

best_val_loss_aug = float('inf')
patience_aug = HYPERPARAMS_AUG["patience"]
patience_counter_aug = 0

for epoch in range(EPOCHS_AUG):
    final_model_aug.train()
    train_loss = 0.0
    for noisy, clean in train_loader_aug:
        noisy, clean = noisy.to(device_aug), clean.to(device_aug)
        optimizer_aug.zero_grad()

        mask_output = final_model_aug(noisy)
        noisy_linear = torch.expm1(noisy)
        predicted_clean_linear = mask_output * noisy_linear
        predicted_clean_log = torch.log1p(torch.clamp(predicted_clean_linear, min=0))
        loss = criterion_aug(predicted_clean_log, clean)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(final_model_aug.parameters(), max_norm=5.0)
        optimizer_aug.step()
        train_loss += loss.item()

    avg_train_loss = train_loss / len(train_loader_aug)
    train_losses_aug.append(avg_train_loss)

    final_model_aug.eval()
    val_loss = 0.0
    with torch.no_grad():
        for noisy, clean in val_loader_aug:
            noisy, clean = noisy.to(device_aug), clean.to(device_aug)
            mask_output = final_model_aug(noisy)
            noisy_linear = torch.expm1(noisy)
            predicted_clean_linear = mask_output * noisy_linear
            predicted_clean_log = torch.log1p(torch.clamp(predicted_clean_linear, min=0))
            loss = criterion_aug(predicted_clean_log, clean)
            val_loss += loss.item()

    avg_val_loss = val_loss / len(val_loader_aug)
    scheduler_aug.step(avg_val_loss)
    val_losses_aug.append(avg_val_loss)

    print(f"Epoch {epoch+1:02d}/{EPOCHS_AUG} - Train Loss: {avg_train_loss:.5f} - Val Loss: {avg_val_loss:.5f}")

    if avg_val_loss < best_val_loss_aug:
        best_val_loss_aug = avg_val_loss
        patience_counter_aug = 0
        torch.save(final_model_aug.state_dict(), "models/best_model_augmented.pth")
    else:
        patience_counter_aug += 1
        if patience_counter_aug >= patience_aug:
            print(f"Early stopping triggered at epoch {epoch+1}")
            break

print(f"\nBest validation loss achieved: {best_val_loss_aug:.4f}")

Epoch 01/40 - Train Loss: 0.03359 - Val Loss: 0.02587
Epoch 02/40 - Train Loss: 0.02512 - Val Loss: 0.02400
Epoch 03/40 - Train Loss: 0.02331 - Val Loss: 0.02155
Epoch 04/40 - Train Loss: 0.02149 - Val Loss: 0.02088
Epoch 05/40 - Train Loss: 0.02059 - Val Loss: 0.01974
Epoch 06/40 - Train Loss: 0.01967 - Val Loss: 0.01871
Epoch 07/40 - Train Loss: 0.01917 - Val Loss: 0.01902
Epoch 08/40 - Train Loss: 0.01856 - Val Loss: 0.01849
Epoch 09/40 - Train Loss: 0.01799 - Val Loss: 0.01786
Epoch 10/40 - Train Loss: 0.01753 - Val Loss: 0.01779
Epoch 11/40 - Train Loss: 0.01698 - Val Loss: 0.01715
Epoch 12/40 - Train Loss: 0.01696 - Val Loss: 0.01666
Epoch 13/40 - Train Loss: 0.01648 - Val Loss: 0.01684
Epoch 14/40 - Train Loss: 0.01651 - Val Loss: 0.01645
Epoch 15/40 - Train Loss: 0.01604 - Val Loss: 0.01691
Epoch 16/40 - Train Loss: 0.01604 - Val Loss: 0.01543
Epoch 17/40 - Train Loss: 0.01595 - Val Loss: 0.01509
Epoch 18/40 - Train Loss: 0.01575 - Val Loss: 0.01566
Epoch 19/40 - Train Loss: 0.

In [7]:
def save_loss_curve_aug(train_loss, val_loss,save_path="outputs/loss_curve_augmented.png"):
    epochs = range(1, len(train_loss) + 1)
    plt.figure(figsize=(8, 5))
    plt.plot(epochs, train_loss, label="Train Loss")
    plt.plot(epochs, val_loss, label="Val Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.title("Training vs Validation Loss (Augmented)")
    plt.legend()
    plt.grid(True)
    plt.savefig(save_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"Saved loss curve to {save_path}")

save_loss_curve_aug(train_losses_aug, val_losses_aug)

Saved loss curve to outputs/loss_curve_augmented.png
